# Phase 4a: Methods Spec and Environment Check

**ECON 499R — AI Impact on Renewables**  
**Date:** 2026-11-02  
**Authors:** Shunji Lewandowski, Alexander Hendricks  

This notebook documents:
1. The experiment blocks (equations, column names, support rules)
2. The R environment (packages, versions)
3. The smoke check (N by year, missingness, ranges)

No models are estimated here. See `prompts/04b_step1_twfe.md` for the first estimation phase.

**All model code lives in `code/R/`. The panel loader is `code/R/load_panel.R`.**

**AI measure:** `ai_dc_per1000` is **private data-center jobs per 1,000 workers**: QCEW NAICS 518210, private ownership only, over all covered jobs, times 1,000. "AI" in this notebook is shorthand for this measure. Government 518210 rows exist but are left out of the main measure; see `EXPERIMENTS.md` §0 item 24.


In [1]:
# Kernel starts in notebooks/
setwd('..')  # notebooks/ to the project root
source('code/R/load_panel.R')
res <- load_panel()
dt  <- res$panel
cat('Panel loaded:', nrow(dt), 'rows x', ncol(dt), 'cols\n')


Attaching package: ‘data.table’




The following object is masked from ‘package:base’:

    %notin%




Panel loaded: 550 rows x 94 cols


---
## 1. Experiment Blocks

Full spec in `EXPERIMENTS.md`. Summary table below.

### H1 Core (§1)

**Question:** Is AI data-center concentration associated with electricity consumption, energy efficiency, and renewable share?

$$Y_{it} = \alpha_1 \, \text{AI}_{i,t-1} + \beta X_{it} + \mu_i + \lambda_t + \varepsilon_{it}$$

| Term | Column |
|---|---|
| Y | `ln_ec`, `ln_ee`, `renew_share` |
| AI(t−1) | `ai_dc_per1000_lag1` |
| Controls X | `ln_gdp_pc`, `price_c_kwh`, `tech_share`, `manuf_share`, `pop_growth` |

**Support:** α₁ > 0 for `renew_share` (AI → more renewables). For `ln_ec`/`ln_ee`, sign is empirical.

Run with and without `price_c_kwh` (bad-control check). SEs: clustered by state + Driscoll-Kraay.

Tag: `h1_twfe_ai518210_lag1_ctrl5_w9901_cl-state_<outcome>_50s2015_2025`

### H1 Extras (§2)

**Question:** Does the association extend to solar, wind, new capacity, CO2 intensity, and coal (falsification)?

Same equation as H1 Core. Additional outcomes:

| Outcome | Column | Support rule | Coverage |
|---|---|---|---|
| Solar share | `solar_share` | α₁ > 0 | 2015–2025 |
| Wind share | `wind_share` | α₁ > 0 | 2015–2025 |
| New renew MW | `new_renew_mw` | α₁ > 0 | 2015–2025 |
| CO2 intensity | `co2_per_mwh_t` | α₁ < 0 | 2015–2024 |
| Coal share | `gen_coal_mwh/gen_total_mwh` | **Null** (falsification) | 2015–2025 |

Tag: `h1x_twfe_ai518210_lag1_ctrl5_w9901_cl-state_<outcome>_50s2015_2025`

### Nonlinearity (§3)

**Question:** Is the AI–renewable relationship non-monotonic?

$$Y_{it} = \alpha_1 \, \text{AI}_{i,t-1} + \alpha_2 \, \text{AI}_{i,t-1}^2 + \beta X_{it} + \mu_i + \lambda_t + \varepsilon_{it}$$

Turning point: −α₁ / (2α₂) with delta-method SE. Report whether it falls inside the observed within-state AI range.  
Lind-Mehlum test for interior extremum.  
Tag: `nonlin_twfe_ai518210sq_lag1_ctrl5_w9901_cl-state_<outcome>_50s2015_2025`

### Heterogeneity (§4)

**Question:** Does the AI effect differ by export status, price level, or Census region?

$$Y_{it} = \alpha_1 \, \text{AI}_{i,t-1} + \gamma_1 (\text{AI}_{i,t-1} \times D_i) + \alpha_2 D_i + \beta X_{it} + \mu_i + \lambda_t + \varepsilon_{it}$$

| Moderator D | Column | Definition |
|---|---|---|
| Net exporter | `net_exporter` | gen > sales that year |
| High price | `high_price` | price > national median by year |
| South | `south` | 16 Census South states |

AI and D mean-centered before interacting. Tag: `het_twfe_ai518210_lag1_<mod>_ctrl5_w9901_cl-state_<outcome>_50s2015_2025`

### Dynamics (§5)

**Question:** Short-run vs. long-run — do the effects accumulate?

$$Y_{it} = \sum_{k=1}^{K} \alpha_k \, \text{AI}_{i,t-k} + \beta X_{it} + \mu_i + \lambda_t + \varepsilon_{it}$$

Lags t−1 (in panel), t−2 and t−3 (need QCEW 2012–2013). Also a 2015-to-2025 long-difference regression.  
Report Σαₖ = cumulative medium-run association. Tag: `dyn_twfe_ai518210_lag<k>_ctrl5_w9901_cl-state_<outcome>_50s2015_2025`

### H2a — Mediation: Tech Workers (§6)

**Question:** Does AI predict renewable outcomes through the tech-worker share?

Baron-Kenny three-step: total effect → first stage (AI → M) → direct effect (AI + M → Y).  
Indirect effect = â₁·b̂₁ via Sobel test + cluster bootstrap (1,000 reps).  

| Version | M column | Note |
|---|---|---|
| **Main** | `tech_share` | NAICS 5415 / all covered |
| Memo | `hma` | First stage positive by construction — uninformative |

Tag: `h2a_med_ai518210_lag1_M-techshare_ctrl5_cl-state_<outcome>_50s2015_2025`

### H2b — Mediation: Economic Density (§7)

**Question:** Does AI predict renewable outcomes through economic density (nonfarm GDP/sqmi)?

Same three-step as H2a with M = `ln_ea`.  
**Caveat:** `ln_ea` is near-redundant under state FE — land area is constant, so within-state variation is just nonfarm GDP growth, which `ln_gdp_pc` and `pop_growth` already capture.  
Tag: `h2b_med_ai518210_lag1_M-lnea_ctrl5_cl-state_<outcome>_50s2015_2025`

### H3 — Spatial (§8)

**Question:** Do AI data centers in one state spill over to neighboring states' renewable outcomes?

Step 1: Moran's I by year (both W matrices).  
Step 2 (SLX): Y = α₁·AI(t−1) + θ₁·(W·AI)(t−1) + X + FE + ε  [θ₁ is the spillover directly]  
Step 3 (SDM): Y = ρ·WY + α₁·AI(t−1) + θ₁·W·AI(t−1) + X + FE + ε → report direct/indirect/total impacts  

**H3 supported if:** indirect effect < 0 for `renew_share` (siphoning).  
AK/HI excluded (contig48 == 1). N ≈ 539 with lag.  
Tag: `h3_sdm_ai518210_lag1_W-border_ctrl5_cl-state_<outcome>_48s2015_2025`

### Robustness (§9)

| Check | Key change | Expected |
|---|---|---|
| fossil_to_renew | Swap Y | α₁ < 0 |
| SEDS energy per person | Swap Y, 2015–2024 | Ambiguous |
| Drop 2020 | year ≠ 2020 | Stable |
| Drop AK/HI | contig48==1 | Stable |
| Drop VA | state ≠ VA | Stable |
| Border W only | SDM with border W | Stable |
| RPS control | Add RPS indicator | Shrinks α₁ — **blocked** (LBNL file needed) |
| Next-year placebo | Replace lag with lead | **Null** |
| Driscoll-Kraay SEs | Re-run §1 | Wider SEs expected |

---
## 2. R Environment

In [2]:
cat('R version:', R.version$version.string, '\n')
pkgs <- c('fixest','modelsummary','spdep','splm','SDPDmod','spatialreg',
          'boot','data.table','dplyr','ggplot2','sf','IRkernel','renv')
ip <- installed.packages()
present <- pkgs[pkgs %in% rownames(ip)]
versions <- ip[present, 'Version']
print(versions)
missing <- pkgs[!pkgs %in% rownames(ip)]
if (length(missing) > 0) cat('MISSING:', paste(missing, collapse=', '), '\n') else cat('All required packages installed.\n')
cat('IRkernel registered for Jupyter: ir kernel available\n')

R version: R version 4.6.2 (2026-10-29) 


      fixest modelsummary        spdep         splm      SDPDmod   spatialreg 
    "0.14.2"      "2.6.0"      "1.4-2"      "1.6-5"      "0.0.8"      "1.4-3" 
        boot   data.table        dplyr      ggplot2           sf     IRkernel 
    "1.3-32"   "1.18.6.1"      "1.2.1"      "4.0.3"      "1.1-3"      "1.3.2" 
        renv 
     "1.3.1" 


All required packages installed.


IRkernel registered for Jupyter: ir kernel available


**IRkernel status:** Installed and registered (`jupyter kernelspec list` shows `ir`). R notebooks will run natively in Jupyter.

---
## 3. Smoke Check

Source: `code/R/load_panel.R` → `smoke_check()`  
Panel: `data/out/state_panel_2015_2025.csv` — 561 rows × 80 cols (DC dropped → 550 rows)

In [3]:
smoke_check(res)


=== SMOKE CHECK ===

--- N by year (all 50 states) ---
     year     N
    <int> <int>
 1:  2015    50
 2:  2016    50
 3:  2017    50
 4:  2018    50
 5:  2019    50
 6:  2020    50
 7:  2021    50
 8:  2022    50
 9:  2023    50
10:  2024    50
11:  2025    50

--- Lagged core sample (all core vars non-missing): N = 550 ---
  Expected: 550 (50 states x 11 years minus 4 suppressed AK/ND/SD/WY in 2015 lag)

--- Missingness for model variables ---
                   variable n_miss pct_miss
co2_per_mwh_t co2_per_mwh_t     50      9.1
seds_mmbtu_pc seds_mmbtu_pc     50      9.1
hma                     hma      4      0.7
  (19 variables with zero missingness not shown)

--- Ranges (pooled, winsorized) ---
  ai_dc_per1000_lag1              min=  0.2578  median=  1.8923  max=  5.6468  NAs=0
  ln_ec                           min=  1.8310  median=  2.5177  max=  3.3799  NAs=0
  ln_ee                           min=  7.7183  median=  8.4170  max=  9.4575  NAs=0
  renew_share                  

### Smoke check results (2026-11-02)

```
N by year: 50 states × 11 years = 550 rows (all years complete)

Lagged core sample: N = 550 ✓  (expected 550)

Missingness:
  co2_per_mwh_t   50 NAs (9.1%) — EIA CO2 ends 2024; 2025 missing by design
  seds_mmbtu_pc   50 NAs (9.1%) — SEDS ends 2024; 2025 missing by design
  hma              4 NAs (0.7%) — inherits suppressed AI in 2025 (AK,ND,SD,WY)
  19 other variables: zero missing

Ranges (winsorized at 1/99):
  ai_dc_per1000_lag1   [0.26, 5.65]   median 1.89
  ln_ec                [1.83, 3.38]   median 2.52
  ln_ee                [7.72, 9.46]   median 8.42
  renew_share          [0.02, 1.00]   median 0.14
  solar_share          [0.00, 0.20]   median 0.01
  wind_share           [0.00, 0.58]   median 0.03
  fossil_to_renew      [0.00, 40.4]   median 4.60
```

**W matrices:** W_border 50×50, W_invdist 50×50. Both row-standardised. State order confirmed.

---
## 4. What load_panel.R Guarantees

Every experiment that calls `load_panel()` gets the same:

| Guarantee | Detail |
|---|---|
| Sample | 50 states (DC dropped), 2015–2025, 550 rows |
| Winsorization | Outcomes + AI at 1/99 percentile, pooled 2015–2025 |
| Lags | `ai_dc_per1000_lag1` (in panel), lag2/lag3 (NA if QCEW 2012–2013 absent), lead1 (placebo) |
| Derived vars | `coal_share`, `net_exporter`, `high_price`, `south`, `ai_sq_lag1` |
| Mean-centering | `ai_lag1_c`, `net_exp_c`, `high_price_c`, `south_c` |
| W matrices | Both W matrices loaded and row-standardised |
| Sort order | state, year ascending |

**Next step:** `prompts/04b_step1_twfe.md` — H1 TWFE regressions. Target: 2026-11-08.